# Bangla Context-Aware Toxicity
## 3 Annotators → IAA → 2-of-3 Joint Majority → No-Majority Blind Queue

This notebook has **one job only**:

1. Load `Annotator1.csv`, `Annotator2.csv`, and `Annotator3.csv`.
2. Verify that the same rows/texts are aligned.
3. Validate the hierarchical labels.
4. Calculate inter-annotator agreement (IAA).
5. Apply **exact 2-of-3 joint hierarchical majority voting**.
6. Save majority-resolved rows separately.
7. Save **only rows with no 2-of-3 joint majority** as one blinded CSV for another annotation round.

It does **not**:
- merge with any previous/new majority dataset;
- create the final test set;
- preprocess the data;
- run EDA.

### Majority rule

Each annotator contributes the complete pair:

`(level_1_class, toxicity_type)`

Examples:

- `Toxic + Insult`, `Toxic + Insult`, `Toxic + Mockery`  
  → majority = `Toxic + Insult`

- `Non-toxic + None`, `Non-toxic + None`, `Toxic + Insult`  
  → majority = `Non-toxic + None`

- `Toxic + Insult`, `Toxic + Mockery`, `Toxic + Harassment`  
  → **NO MAJORITY** → sent to new annotators

- `Non-toxic + None`, `Toxic + Insult`, `Toxic + Mockery`  
  → **NO MAJORITY** → sent to new annotators

The no-majority file hides all original labels.

## 0. Configuration

In [4]:
# ============================== CONFIG =====================================
SEED = 42
USE_DRIVE = True

PROJECT_ROOT = "/content/drive/MyDrive/toxicity"

TEST_SET_SUBDIR = "test set"
ANNOTATOR_SUBDIR = "annotator"
NEW_ANNOTATION_SUBDIR = "independently annotate"

ANNOTATOR_FILES = {
    "Annotator1": "Annotator1.csv",
    "Annotator2": "Annotator2.csv",
    "Annotator3": "Annotator3.csv",
}

IAA_OUTPUT_SUBDIR = "iaa_results"

MAJORITY_RESOLVED_NAME = "THREE_ANNOTATOR_MAJORITY_RESOLVED.csv"
MAJORITY_AUDIT_NAME = "THREE_ANNOTATOR_MAJORITY_WITH_AUDIT.csv"

NO_MAJORITY_BLIND_NAME = "NO_MAJORITY_BLIND_FOR_NEW_ANNOTATORS.csv"
NO_MAJORITY_PRIVATE_AUDIT_NAME = "NO_MAJORITY_PRIVATE_AUDIT.csv"

# Safety only. This does NOT define what is blind.
# If >60% of the entire dataset has no 2-of-3 JOINT majority,
# something is probably wrong with the files/schema and execution stops.
MAX_NO_MAJORITY_SHARE = 0.60
# ==========================================================================

from pathlib import Path
import os, sys

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path(PROJECT_ROOT)
TEST_SET_DIR = ROOT / TEST_SET_SUBDIR
ANNOT_DIR = TEST_SET_DIR / ANNOTATOR_SUBDIR
IAA_DIR = ANNOT_DIR / IAA_OUTPUT_SUBDIR
NEW_ANNOTATION_DIR = TEST_SET_DIR / NEW_ANNOTATION_SUBDIR

IAA_DIR.mkdir(parents=True, exist_ok=True)
NEW_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

MAJORITY_RESOLVED_PATH = ANNOT_DIR / MAJORITY_RESOLVED_NAME
MAJORITY_AUDIT_PATH = IAA_DIR / MAJORITY_AUDIT_NAME

NO_MAJORITY_BLIND_PATH = NEW_ANNOTATION_DIR / NO_MAJORITY_BLIND_NAME
NO_MAJORITY_PRIVATE_AUDIT_PATH = IAA_DIR / NO_MAJORITY_PRIVATE_AUDIT_NAME

print("Annotator folder       :", ANNOT_DIR)
print("IAA output folder      :", IAA_DIR)
print("New annotation folder  :", NEW_ANNOTATION_DIR)
print("No-majority blind file :", NO_MAJORITY_BLIND_PATH)

Mounted at /content/drive
Annotator folder       : /content/drive/MyDrive/toxicity/test set/annotator
IAA output folder      : /content/drive/MyDrive/toxicity/test set/annotator/iaa_results
New annotation folder  : /content/drive/MyDrive/toxicity/test set/independently annotate
No-majority blind file : /content/drive/MyDrive/toxicity/test set/independently annotate/NO_MAJORITY_BLIND_FOR_NEW_ANNOTATORS.csv


## 1. Imports, taxonomy, and helper functions

In [5]:
import json
import re
import hashlib
import unicodedata
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score

np.random.seed(SEED)

LEVEL1_LABELS = ["Non-toxic", "Toxic"]

LEVEL2_LABELS = [
    "Insult",
    "Threat",
    "Hate Speech",
    "Profanity",
    "Harassment",
    "Identity Attack",
    "Toxic Endorsement / Agreement",
    "Incitement / Encouragement of Harm",
    "Mockery / Ridicule",
    "Sexual Harassment / Objectification",
    "Dehumanization",
]

HIER_LABELS = ["None"] + LEVEL2_LABELS

def read_csv_preserve_none(path):
    return pd.read_csv(path, keep_default_na=False)

def write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved -> {path}")

def normalize_text(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\u200B-\u200D\u2060\uFEFF]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_level1(value):
    s = str(value).strip().lower().replace("_", "-")
    mapping = {
        "0": "Non-toxic",
        "non-toxic": "Non-toxic",
        "non toxic": "Non-toxic",
        "nontoxic": "Non-toxic",
        "1": "Toxic",
        "toxic": "Toxic",
    }
    return mapping.get(s, str(value).strip())

def normalize_level2(value):
    s = str(value).strip()

    if s == "" or s.lower() in {"none", "nan", "null", "na", "n/a"}:
        return "None"

    aliases = {
        "Mockery/Ridicule": "Mockery / Ridicule",
        "Mockery & Ridicule": "Mockery / Ridicule",
        "Sexual Harassment/Objectification": "Sexual Harassment / Objectification",
        "Sexual Harassment & Objectification": "Sexual Harassment / Objectification",
        "Toxic Endorsement/Agreement": "Toxic Endorsement / Agreement",
        "Toxic Endorsement & Agreement": "Toxic Endorsement / Agreement",
        "Incitement/Encouragement of Harm": "Incitement / Encouragement of Harm",
        "Incitement & Encouragement of Harm": "Incitement / Encouragement of Harm",
    }
    return aliases.get(s, s)

def row_key(source, context, reply):
    payload = "\u241E".join([
        normalize_text(source).lower(),
        normalize_text(context),
        normalize_text(reply),
    ])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def validate_hierarchy(df, name):
    invalid_l1 = ~df["level_1_class"].isin(LEVEL1_LABELS)
    invalid_l2 = ~df["toxicity_type"].isin(HIER_LABELS)

    bad_non_toxic = (
        df["level_1_class"].eq("Non-toxic")
        & ~df["toxicity_type"].eq("None")
    )

    bad_toxic = (
        df["level_1_class"].eq("Toxic")
        & ~df["toxicity_type"].isin(LEVEL2_LABELS)
    )

    problems = {
        "invalid_level1": int(invalid_l1.sum()),
        "invalid_level2": int(invalid_l2.sum()),
        "non_toxic_with_toxic_subtype": int(bad_non_toxic.sum()),
        "toxic_without_valid_subtype": int(bad_toxic.sum()),
    }

    if any(problems.values()):
        raise ValueError(f"{name} hierarchy error: {problems}")

    print(f"{name}: hierarchy PASS")

## 2. Load and align the three annotator files

In [6]:
annotators = {}

for name, filename in ANNOTATOR_FILES.items():
    path = ANNOT_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = read_csv_preserve_none(path)

    required = [
        "Source",
        "context",
        "reply",
        "level_1_class",
        "toxicity_type",
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{filename} missing columns: {missing}")

    df = df.copy()
    df["level_1_class"] = df["level_1_class"].map(normalize_level1)
    df["toxicity_type"] = df["toxicity_type"].map(normalize_level2)

    validate_hierarchy(df, name)

    df["_row_key"] = [
        row_key(s, c, r)
        for s, c, r in zip(df["Source"], df["context"], df["reply"])
    ]

    annotators[name] = df

lengths = {name: len(df) for name, df in annotators.items()}
print("Rows per annotator:", lengths)

if len(set(lengths.values())) != 1:
    raise ValueError("Annotator row counts differ.")

names = list(ANNOTATOR_FILES.keys())
reference = annotators[names[0]]

for other_name in names[1:]:
    other = annotators[other_name]

    key_match = reference["_row_key"].equals(other["_row_key"])

    if not key_match:
        mismatch = reference["_row_key"].ne(other["_row_key"])
        bad_rows = np.where(mismatch.to_numpy())[0][:20] + 1
        raise ValueError(
            f"Row/text alignment failed for {other_name}. "
            f"First mismatched row numbers: {bad_rows.tolist()}"
        )

print("Row/text alignment across all 3 annotators: PASS")

duplicate_identity_rows = int(reference["_row_key"].duplicated().sum())
print("Exact duplicate item identities present:", duplicate_identity_rows)
print(
    "Note: duplicates are reported but NOT silently removed; "
    "majority voting remains row-preserving."
)

Annotator1: hierarchy PASS
Annotator2: hierarchy PASS
Annotator3: hierarchy PASS
Rows per annotator: {'Annotator1': 2330, 'Annotator2': 2330, 'Annotator3': 2330}
Row/text alignment across all 3 annotators: PASS
Exact duplicate item identities present: 6
Note: duplicates are reported but NOT silently removed; majority voting remains row-preserving.


## 3. IAA metrics

The notebook reports three views:

### Level-1 IAA
All rows:
- `Non-toxic`
- `Toxic`

### Conditional Level-2 IAA
A subtype rating is included only when that annotator marked the row `Toxic`.

Rows need at least two valid toxic-subtype ratings to contribute to multi-rater Level-2 metrics.

### Joint hierarchical IAA
Each annotation is one of 12 complete states:
- `None` for Non-toxic
- one of 11 toxic subtypes for Toxic

This is the same representation used for majority voting.

In [7]:
def code_matrix_level1():
    mapping = {label: i for i, label in enumerate(LEVEL1_LABELS)}
    return np.array([
        [mapping[annotators[a].iloc[i]["level_1_class"]] for a in names]
        for i in range(len(reference))
    ], dtype=np.int16)

def code_matrix_level2_conditional():
    mapping = {label: i for i, label in enumerate(LEVEL2_LABELS)}
    codes = np.full((len(reference), len(names)), -1, dtype=np.int16)

    for j, a in enumerate(names):
        for i, row in annotators[a].iterrows():
            if row["level_1_class"] == "Toxic":
                codes[i, j] = mapping[row["toxicity_type"]]

    return codes

def code_matrix_joint():
    mapping = {label: i for i, label in enumerate(HIER_LABELS)}
    return np.array([
        [mapping[annotators[a].iloc[i]["toxicity_type"]] for a in names]
        for i in range(len(reference))
    ], dtype=np.int16)

C1 = code_matrix_level1()
C2 = code_matrix_level2_conditional()
CJ = code_matrix_joint()

print("Level-1 matrix :", C1.shape)
print("Level-2 matrix :", C2.shape)
print("Joint matrix   :", CJ.shape)

Level-1 matrix : (2330, 3)
Level-2 matrix : (2330, 3)
Joint matrix   : (2330, 3)


In [8]:
def rating_count_matrix(codes, n_categories):
    counts = np.zeros((codes.shape[0], n_categories), dtype=np.int64)
    for j in range(n_categories):
        counts[:, j] = (codes == j).sum(axis=1)
    return counts

def pairwise_mean_agreement(codes, n_categories):
    counts = rating_count_matrix(codes, n_categories)
    n_raters = counts.sum(axis=1)
    keep = n_raters >= 2

    counts = counts[keep]
    n_raters = n_raters[keep]

    P_i = (
        np.square(counts).sum(axis=1) - n_raters
    ) / (
        n_raters * (n_raters - 1)
    )

    return float(P_i.mean()), int(keep.sum())

def fleiss_kappa(codes, n_categories):
    counts = rating_count_matrix(codes, n_categories)
    n_raters = counts.sum(axis=1)
    keep = n_raters >= 2

    counts = counts[keep]
    n_raters = n_raters[keep]

    P_i = (
        np.square(counts).sum(axis=1) - n_raters
    ) / (
        n_raters * (n_raters - 1)
    )

    P_bar = float(P_i.mean())

    p_j = counts.sum(axis=0) / counts.sum()
    P_e = float(np.square(p_j).sum())

    if np.isclose(P_e, 1):
        return np.nan

    return float((P_bar - P_e) / (1 - P_e))

def gwet_ac1(codes, n_categories):
    counts = rating_count_matrix(codes, n_categories)
    n_raters = counts.sum(axis=1)
    keep = n_raters >= 2

    counts = counts[keep]
    n_raters = n_raters[keep]

    P_i = (
        np.square(counts).sum(axis=1) - n_raters
    ) / (
        n_raters * (n_raters - 1)
    )

    P_bar = float(P_i.mean())

    p_j = counts.sum(axis=0) / counts.sum()
    P_e = float(
        (p_j * (1 - p_j)).sum() / (n_categories - 1)
    )

    if np.isclose(P_e, 1):
        return np.nan

    return float((P_bar - P_e) / (1 - P_e))

def krippendorff_alpha_nominal(codes, n_categories):
    coincidence = np.zeros(
        (n_categories, n_categories),
        dtype=float,
    )

    for row in codes:
        vals = row[row >= 0]
        m = len(vals)

        if m < 2:
            continue

        cnt = np.bincount(vals, minlength=n_categories).astype(float)

        outer = np.outer(cnt, cnt)
        np.fill_diagonal(outer, cnt * (cnt - 1))

        coincidence += outer / (m - 1)

    marg = coincidence.sum(axis=1)
    n = marg.sum()

    if n <= 1:
        return np.nan

    Do = (
        coincidence.sum() - np.trace(coincidence)
    ) / n

    De = (
        n * n - np.square(marg).sum()
    ) / (
        n * (n - 1)
    )

    if np.isclose(De, 0):
        return np.nan

    return float(1 - Do / De)

def overall_metric_row(label, codes, categories):
    agreement, usable = pairwise_mean_agreement(
        codes, len(categories)
    )

    return {
        "analysis": label,
        "total_rows": len(codes),
        "usable_rows": usable,
        "pairwise_mean_agreement": agreement,
        "fleiss_kappa": fleiss_kappa(codes, len(categories)),
        "gwet_ac1": gwet_ac1(codes, len(categories)),
        "krippendorff_alpha_nominal": krippendorff_alpha_nominal(
            codes, len(categories)
        ),
    }

overall_iaa = pd.DataFrame([
    overall_metric_row(
        "Level-1 binary",
        C1,
        LEVEL1_LABELS,
    ),
    overall_metric_row(
        "Conditional Level-2 toxic subtype",
        C2,
        LEVEL2_LABELS,
    ),
    overall_metric_row(
        "Joint hierarchical 12-class",
        CJ,
        HIER_LABELS,
    ),
])

display(overall_iaa)
write_csv(
    overall_iaa,
    IAA_DIR / "overall_iaa.csv",
)

,analysis,total_rows,usable_rows,pairwise_mean_agreement,fleiss_kappa,gwet_ac1,krippendorff_alpha_nominal
0,Level-1 binary,2330,2330,0.910157,0.818420,0.822170,0.818446
1,Conditional Level-2 toxic subtype,2330,1346,0.671620,0.562778,0.644953,0.556886
2,Joint hierarchical 12-class,2330,2330,0.739056,0.639312,0.720685,0.639364


Saved -> /content/drive/MyDrive/toxicity/test set/annotator/iaa_results/overall_iaa.csv


## 4. Pairwise Cohen's κ and raw agreement

In [9]:
pairwise_rows = []

for a, b in combinations(names, 2):

    # Level-1
    xa = annotators[a]["level_1_class"]
    xb = annotators[b]["level_1_class"]

    pairwise_rows.append({
        "analysis": "Level-1",
        "annotator_a": a,
        "annotator_b": b,
        "n": len(reference),
        "raw_agreement": float((xa == xb).mean()),
        "cohen_kappa": float(cohen_kappa_score(xa, xb)),
    })

    # Conditional Level-2: only rows where BOTH called Toxic.
    mask = (
        annotators[a]["level_1_class"].eq("Toxic")
        & annotators[b]["level_1_class"].eq("Toxic")
    )

    ya = annotators[a].loc[mask, "toxicity_type"]
    yb = annotators[b].loc[mask, "toxicity_type"]

    if len(ya):
        pairwise_rows.append({
            "analysis": "Conditional Level-2 (both Toxic)",
            "annotator_a": a,
            "annotator_b": b,
            "n": int(mask.sum()),
            "raw_agreement": float((ya == yb).mean()),
            "cohen_kappa": float(cohen_kappa_score(ya, yb)),
        })

    # Joint hierarchical label
    ja = annotators[a]["toxicity_type"]
    jb = annotators[b]["toxicity_type"]

    pairwise_rows.append({
        "analysis": "Joint hierarchical 12-class",
        "annotator_a": a,
        "annotator_b": b,
        "n": len(reference),
        "raw_agreement": float((ja == jb).mean()),
        "cohen_kappa": float(cohen_kappa_score(ja, jb)),
    })

pairwise_iaa = pd.DataFrame(pairwise_rows)
display(pairwise_iaa)

write_csv(
    pairwise_iaa,
    IAA_DIR / "pairwise_cohen_kappa.csv",
)

,analysis,annotator_a,annotator_b,n,raw_agreement,cohen_kappa
0,Level-1,Annotator1,Annotator2,2330,0.987124,0.973605
1,Conditional Level-2 (both Toxic),Annotator1,Annotator2,1332,0.791291,0.711598
2,Joint hierarchical 12-class,Annotator1,Annotator2,2330,0.867811,0.818859
3,Level-1,Annotator1,Annotator3,2330,0.869528,0.739299
4,Conditional Level-2 (both Toxic),Annotator1,Annotator3,1100,0.619091,0.496903
5,Joint hierarchical 12-class,Annotator1,Annotator3,2330,0.689700,0.571473
6,Level-1,Annotator2,Annotator3,2330,0.873820,0.747877
7,Conditional Level-2 (both Toxic),Annotator2,Annotator3,1106,0.548825,0.426008
8,Joint hierarchical 12-class,Annotator2,Annotator3,2330,0.659657,0.534951


Saved -> /content/drive/MyDrive/toxicity/test set/annotator/iaa_results/pairwise_cohen_kappa.csv


## 5. Exact 2-of-3 JOINT majority voting

This is the critical section.

The notebook does **not** send every disagreement to new annotators.

A row is sent to new annotators **only if no complete joint label receives at least two votes**.

In [10]:
majority_rows = []
no_majority_rows = []

for i in range(len(reference)):
    ref = reference.iloc[i]

    votes = [
        (
            annotators[a].iloc[i]["level_1_class"],
            annotators[a].iloc[i]["toxicity_type"],
        )
        for a in names
    ]

    vote_counts = Counter(votes)
    winner, winner_count = vote_counts.most_common(1)[0]

    common = {
        "original_row_number": i + 1,
        "Source": ref["Source"],
        "context": ref["context"],
        "reply": ref["reply"],
        "_row_key": ref["_row_key"],
    }

    vote_string = " | ".join(
        f"{names[j]}={votes[j][0]}::{votes[j][1]}"
        for j in range(3)
    )

    if winner_count >= 2:
        majority_rows.append({
            **common,
            "level_1_class": winner[0],
            "toxicity_type": winner[1],
            "majority_vote_count": int(winner_count),
            "majority_pattern": "3_of_3" if winner_count == 3 else "2_of_3",
            "private_vote_pattern": vote_string,
        })
    else:
        no_majority_rows.append({
            **common,
            "Annotator1_level_1": votes[0][0],
            "Annotator1_toxicity_type": votes[0][1],
            "Annotator2_level_1": votes[1][0],
            "Annotator2_toxicity_type": votes[1][1],
            "Annotator3_level_1": votes[2][0],
            "Annotator3_toxicity_type": votes[2][1],
            "private_vote_pattern": vote_string,
        })

majority_resolved = pd.DataFrame(majority_rows)
no_majority_private = pd.DataFrame(no_majority_rows)

n_total = len(reference)
n_resolved = len(majority_resolved)
n_no_majority = len(no_majority_private)
no_majority_share = n_no_majority / max(n_total, 1)

print("=" * 72)
print("MAJORITY VOTING SUMMARY")
print("=" * 72)
print(f"Total rows           : {n_total:,}")
print(f"Majority resolved    : {n_resolved:,}")
print(f"No 2-of-3 majority   : {n_no_majority:,}")
print(f"No-majority share    : {no_majority_share:.2%}")

if no_majority_share > MAX_NO_MAJORITY_SHARE:
    raise RuntimeError(
        f"SAFETY STOP: {no_majority_share:.2%} of rows have no joint majority. "
        "This is unusually high. Check that the correct three annotator files "
        "were loaded and aligned."
    )

MAJORITY VOTING SUMMARY
Total rows           : 2,330
Majority resolved    : 2,142
No 2-of-3 majority   : 188
No-majority share    : 8.07%


## 6. Export majority-resolved rows

In [11]:
majority_clean = majority_resolved[
    [
        "Source",
        "context",
        "reply",
        "level_1_class",
        "toxicity_type",
    ]
].copy()

validate_hierarchy(
    majority_clean,
    "Majority-resolved dataset",
)

write_csv(
    majority_clean,
    MAJORITY_RESOLVED_PATH,
)

write_csv(
    majority_resolved,
    MAJORITY_AUDIT_PATH,
)

print("Resolved majority file:")
print(MAJORITY_RESOLVED_PATH)

Majority-resolved dataset: hierarchy PASS
Saved -> /content/drive/MyDrive/toxicity/test set/annotator/THREE_ANNOTATOR_MAJORITY_RESOLVED.csv
Saved -> /content/drive/MyDrive/toxicity/test set/annotator/iaa_results/THREE_ANNOTATOR_MAJORITY_WITH_AUDIT.csv
Resolved majority file:
/content/drive/MyDrive/toxicity/test set/annotator/THREE_ANNOTATOR_MAJORITY_RESOLVED.csv


## 7. Create ONE blind CSV only for NO-MAJORITY rows

This file is the one to send to another group of annotators.

The new annotators do **not** receive:
- Annotator1/2/3 labels;
- vote patterns;
- IAA results;
- disagreement type;
- majority information.

The rows are shuffled before blind IDs are assigned.

In [12]:
if len(no_majority_private) == 0:
    blind = pd.DataFrame(
        columns=[
            "blind_item_id",
            "Source",
            "context",
            "reply",
            "level_1_class",
            "toxicity_type",
            "annotator_notes",
        ]
    )

else:
    # Shuffle so original row positions are hidden.
    shuffled = no_majority_private.sample(
        frac=1.0,
        random_state=SEED,
    ).reset_index(drop=True)

    shuffled.insert(
        0,
        "blind_item_id",
        [
            f"BLIND_{i:04d}"
            for i in range(1, len(shuffled) + 1)
        ],
    )

    blind = shuffled[
        [
            "blind_item_id",
            "Source",
            "context",
            "reply",
        ]
    ].copy()

    blind["level_1_class"] = ""
    blind["toxicity_type"] = ""
    blind["annotator_notes"] = ""

    # Private audit gets the same blind_item_id so returned files can be mapped later.
    private_audit = shuffled[
        [
            "blind_item_id",
            "original_row_number",
            "Source",
            "context",
            "reply",
            "Annotator1_level_1",
            "Annotator1_toxicity_type",
            "Annotator2_level_1",
            "Annotator2_toxicity_type",
            "Annotator3_level_1",
            "Annotator3_toxicity_type",
            "private_vote_pattern",
            "_row_key",
        ]
    ].copy()

    write_csv(
        private_audit,
        NO_MAJORITY_PRIVATE_AUDIT_PATH,
    )

write_csv(
    blind,
    NO_MAJORITY_BLIND_PATH,
)

print("=" * 72)
print("BLIND NO-MAJORITY FILE CREATED")
print("=" * 72)
print("Rows:", len(blind))
print("Give ONLY this file to the new annotators:")
print(NO_MAJORITY_BLIND_PATH)
print()
print("Keep this PRIVATE:")
print(NO_MAJORITY_PRIVATE_AUDIT_PATH)

Saved -> /content/drive/MyDrive/toxicity/test set/annotator/iaa_results/NO_MAJORITY_PRIVATE_AUDIT.csv
Saved -> /content/drive/MyDrive/toxicity/test set/independently annotate/NO_MAJORITY_BLIND_FOR_NEW_ANNOTATORS.csv
BLIND NO-MAJORITY FILE CREATED
Rows: 188
Give ONLY this file to the new annotators:
/content/drive/MyDrive/toxicity/test set/independently annotate/NO_MAJORITY_BLIND_FOR_NEW_ANNOTATORS.csv

Keep this PRIVATE:
/content/drive/MyDrive/toxicity/test set/annotator/iaa_results/NO_MAJORITY_PRIVATE_AUDIT.csv


## 8. Agreement-pattern and majority summary tables

In [13]:
summary = pd.DataFrame([
    {
        "measure": "total_rows",
        "value": len(reference),
    },
    {
        "measure": "3_of_3_joint_majority",
        "value": int(
            majority_resolved["majority_vote_count"].eq(3).sum()
        ),
    },
    {
        "measure": "2_of_3_joint_majority",
        "value": int(
            majority_resolved["majority_vote_count"].eq(2).sum()
        ),
    },
    {
        "measure": "no_joint_majority",
        "value": len(no_majority_private),
    },
    {
        "measure": "no_joint_majority_percent",
        "value": 100 * len(no_majority_private) / max(len(reference), 1),
    },
    {
        "measure": "exact_duplicate_item_rows_reported",
        "value": duplicate_identity_rows,
    },
])

display(summary)

write_csv(
    summary,
    IAA_DIR / "majority_summary.csv",
)

print("\nMajority-resolved Level-1 distribution:")
display(
    majority_clean["level_1_class"]
    .value_counts()
    .rename_axis("level_1_class")
    .reset_index(name="count")
)

print("\nMajority-resolved Level-2 distribution:")
display(
    majority_clean["toxicity_type"]
    .value_counts()
    .rename_axis("toxicity_type")
    .reset_index(name="count")
)

,measure,value
0,total_rows,2330.00000
1,3_of_3_joint_majority,1512.00000
2,2_of_3_joint_majority,630.00000
3,no_joint_majority,188.00000
4,no_joint_majority_percent,8.06867
5,exact_duplicate_item_rows_reported,6.00000


Saved -> /content/drive/MyDrive/toxicity/test set/annotator/iaa_results/majority_summary.csv

Majority-resolved Level-1 distribution:


,level_1_class,count
0,Toxic,1158
1,Non-toxic,984



Majority-resolved Level-2 distribution:


,toxicity_type,count
0,None,984
1,Mockery / Ridicule,447
2,Insult,401
3,Harassment,111
4,Profanity,73
5,Sexual Harassment / Objectification,38
6,Hate Speech,24
7,Dehumanization,23
8,Incitement / Encouragement of Harm,15
9,Identity Attack,11


## 9. Final folder result

After running the notebook:

```text
MyDrive/
└── toxicity/
    └── test set/
        ├── annotator/
        │   ├── Annotator1.csv
        │   ├── Annotator2.csv
        │   ├── Annotator3.csv
        │   ├── THREE_ANNOTATOR_MAJORITY_RESOLVED.csv
        │   └── iaa_results/
        │       ├── overall_iaa.csv
        │       ├── pairwise_cohen_kappa.csv
        │       ├── THREE_ANNOTATOR_MAJORITY_WITH_AUDIT.csv
        │       ├── NO_MAJORITY_PRIVATE_AUDIT.csv
        │       └── majority_summary.csv
        │
        └── independently annotate/
            └── NO_MAJORITY_BLIND_FOR_NEW_ANNOTATORS.csv
```

### Important

`NO_MAJORITY_BLIND_FOR_NEW_ANNOTATORS.csv` contains **only true 3-way joint-label conflicts** where no joint label received two votes.

Do not give `NO_MAJORITY_PRIVATE_AUDIT.csv` to the new annotators.